# Forecast Ver3 - Peak Gate Notebook

Notebook version of `forecast_ver3_peakgate.py`.

## 1) Run Ver3 pipeline

In [4]:
import runpy
from pathlib import Path

script = Path('forecast_ver3_peakgate.py')
if not script.exists():
    script = Path('src_3') / 'forecast_ver3_peakgate.py
'
runpy.run_path(str(script), run_name='__main__')

SyntaxError: unterminated string literal (detected at line 6) (2588314347.py, line 6)

## 2) Load test predictions and quick metrics

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

root = Path.cwd()
if not (root / 'outputs').exists():
    root = root.parent

pred_path = root / 'outputs' / 'forecast_ver3' / 'tables' / 'test_predictions_ver3.csv'
df = pd.read_csv(pred_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

def m(y, p):
    return {
        'MAE': float(mean_absolute_error(y, p)),
        'RMSE': float(np.sqrt(mean_squared_error(y, p))),
        'R2': float(r2_score(y, p)),
        'Bias': float(np.mean(p - y)),
    }

all_metrics = m(df['Revenue'].to_numpy(), df['Revenue_pred'].to_numpy())
all_metrics

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Project\\Personal Project\\Datathon 2026\\src_3\\outputs\\forecast_ver3\\tables\\test_predictions_ver3.csv'

## 3) Segment diagnostics (peak and fast-growth)

In [ ]:
q90, q95 = df['Revenue'].quantile([0.90, 0.95])
local_peak = ((df['Revenue'] > df['Revenue'].shift(1)) & (df['Revenue'] > df['Revenue'].shift(-1)) & (df['Revenue'] >= q90)).fillna(False)
fast_growth = (df['Revenue'].diff() >= df['Revenue'].diff()[df['Revenue'].diff() > 0].quantile(0.90)).fillna(False)

rows = []
for name, mask in [('all', pd.Series([True]*len(df))), ('top10', df['Revenue'] >= q90), ('top5', df['Revenue'] >= q95), ('local_peak_p90', local_peak), ('fast_growth_top10', fast_growth)]:
    y = df.loc[mask, 'Revenue'].to_numpy()
    p = df.loc[mask, 'Revenue_pred'].to_numpy()
    rows.append({
        'segment': name,
        'n': int(mask.sum()),
        'MAE': mean_absolute_error(y, p),
        'RMSE': np.sqrt(mean_squared_error(y, p)),
        'R2': r2_score(y, p),
        'Bias': np.mean(p - y),
        'Pred/Actual': np.mean(p / y),
        'UnderPredRate': np.mean(p < y),
    })

pd.DataFrame(rows)

## 4) Plot actual vs pred

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,5))
plt.plot(df['Date'], df['Revenue'], lw=0.9, label='Actual Revenue')
plt.plot(df['Date'], df['Revenue_pred'], lw=0.85, label='Ver3 Revenue')
plt.plot(df['Date'], df['Revenue_base_log_xgb'], lw=0.75, alpha=0.7, label='Base log-XGB')
plt.title('Ver3 Test Revenue')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.tight_layout()
plt.show()

## 5) Top under-predicted days

In [ ]:
tmp = df[['Date','Revenue','Revenue_pred']].copy()
tmp['error'] = tmp['Revenue_pred'] - tmp['Revenue']
tmp['pred_over_actual'] = tmp['Revenue_pred'] / tmp['Revenue']
tmp.sort_values('error').head(20)